# 01 — Đọc và kiểm tra dữ liệu: chuẩn bị Google Colab
## 1. Mục đích
**P0-T3 · Research Demo / REES46 · READY FOR STUDENT TEST**

Google Colab là nơi chạy notebook Python qua trình duyệt. Google Drive giúp giữ tệp sau khi phiên chạy kết thúc; Jupyter local chạy trên Python và tài nguyên máy của bạn.

Câu hỏi của bài này: **Nhóm có đọc được đúng tệp đã kiểm kê và giải thích được kết quả không?**

Chúng ta chỉ kiểm tra môi trường và tệp `buyer_features.parquet` (~2,16 MiB). Đây là **bảng ứng viên**, chưa được nghiệm thu làm dữ liệu nghiên cứu. Không cần tải khoảng 20 triệu events lên Drive. Không EDA, vẽ biểu đồ, tạo đặc trưng hoặc chạy K-Means.

**Cách học:** đọc ô Markdown trước ô Python, chạy theo thứ tự. Sau khi hoàn thành, khởi động lại phiên và chạy toàn bộ từ đầu để tránh phụ thuộc biến của lần trước. Không chia sẻ output chứa dữ liệu cá nhân.


## 2. Kiểm tra môi trường Python
Một notebook có ô **Markdown** để giải thích và ô **Python** để chạy code. Nhấn nút chạy hoặc Shift+Enter để chạy một ô.

Bài này chỉ cần DuckDB và thư viện chuẩn Python; chưa dùng pandas, PyArrow, Streamlit hoặc scikit-learn. Ô dưới chỉ cài DuckDB nếu chưa tìm thấy. Nếu thư viện đã cài nhưng import lỗi, hãy đọc lỗi thay vì tự đổi chính sách bảo mật.

Đầu ra cần hiểu: phiên bản Python, DuckDB và đang chạy Colab hay local. Việc import được DuckDB chưa xác nhận toàn bộ môi trường MVP.


In [ ]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])

import duckdb

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

print(json.dumps({
    "python": sys.version.split()[0],
    "duckdb": duckdb.__version__,
    "running_in_colab": IN_COLAB,
}, ensure_ascii=False, indent=2))


## 3. Kết nối Google Drive và chọn dữ liệu
**Chỉ trên Colab:** ô dưới yêu cầu quyền qua giao diện Google. Chọn đúng tài khoản của bạn và tự xác nhận quyền. Không dán mật khẩu, token hoặc mã đăng nhập vào notebook. Chạy local sẽ bỏ qua mount Drive.

Tự chuẩn bị đúng một tệp `buyer_features.parquet` từ bản đã kiểm kê nếu nhóm đồng ý đưa tệp lên Drive; không đổi quyền chia sẻ công khai. Vị trí đề xuất: `MyDrive/datasets/rees46/features/buyer_features.parquet`. Notebook không upload, copy hoặc thay đổi quyền của bất kỳ tệp nào.


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Đang chạy local: bỏ qua kết nối Google Drive.")


### Chỉ chỉnh một biến DATA_ROOT
Trên Colab, giá trị mặc định trỏ tới thư mục `rees46` đề xuất trong Drive. Nếu bạn đặt file nơi khác, sửa **DATA_ROOT** thành thư mục chứa `features/`.

Trên local, có thể đặt biến môi trường `REES46_DATA_ROOT` trỏ tới thư mục dữ liệu có sẵn. Nếu không đặt, mặc định là `data/` trong thư mục làm việc hiện tại. Không cần ghi đường dẫn máy Windows vào notebook. Đây là notebook tự chạy, không cần clone repo hoặc import helper của dự án.


In [ ]:
DATA_ROOT = Path(os.environ.get(
    "REES46_DATA_ROOT",
    "/content/drive/MyDrive/datasets/rees46" if IN_COLAB else "data",
)).expanduser()

DATA_FILE = DATA_ROOT / "features" / "buyer_features.parquet"
print("Tệp cần đọc:", DATA_FILE.name)
print("Đã tìm thấy tệp:", DATA_FILE.is_file())


## 4. Đọc thử bằng DuckDB
**Schema** là tên và kiểu của các cột; **row count** là số dòng. Một dòng của bảng này đại diện cho một khách hàng, không phải một event hoặc một đơn hàng.

Ô đầu dừng rõ ràng nếu thiếu file. Khi đó kiểm tra mount Drive, DATA_ROOT và thư mục `features/`; không tạo dữ liệu giả hoặc bỏ qua lỗi để báo đạt.


In [ ]:
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        "Thiếu features/buyer_features.parquet. "
        "Hãy chuẩn bị đúng tệp ứng viên đã kiểm kê, kiểm tra DATA_ROOT "
        "và kết nối Drive rồi chạy lại từ đầu. Không tạo dữ liệu giả."
    )

print("Đã tìm thấy tệp ứng viên; bắt đầu đọc cấu trúc bảng.")


### Xem cấu trúc, không xem mã khách hàng
DuckDB đọc Parquet trực tiếp. Chúng ta chỉ in tên cột/kiểu dữ liệu; không hiển thị dòng khách hàng hay user IDs.

Bảng có thể chứa 23 cột, nhưng mô hình MVP sau này chỉ dùng đúng năm đặc trưng đã chọn. Cột `user_id` là khóa để nối bảng, không phải đặc trưng phân cụm.


In [ ]:
con = duckdb.connect(":memory:", config={"threads": 2, "memory_limit": "512MB"})
buyers = con.read_parquet(str(DATA_FILE))
_ = buyers.create_view("buyers")

schema_rows = con.execute("DESCRIBE buyers").fetchall()
actual_schema = {row[0]: row[1] for row in schema_rows}
for name, data_type in actual_schema.items():
    print(f"{name}: {data_type}")

required_columns = {
    "user_id", "is_buyer", "recency_days", "frequency", "monetary",
    "sessions_30d", "views_30d",
}
missing_columns = sorted(required_columns - actual_schema.keys())
if missing_columns:
    raise ValueError("Thiếu cột cần thiết: " + ", ".join(missing_columns))


### Đếm một số thông tin để kiểm tra tệp
RFM gồm Recency (thời gian từ lần mua cuối đến mốc chốt), Frequency (số phiên mua riêng biệt) và Monetary (tổng giá trị mua hợp lệ). sessions_30d/views_30d là số phiên/lượt xem trong 30 ngày.

Ô dưới chỉ tính các số đếm phục vụ kiểm tra đọc file: số dòng, số user riêng biệt, số dòng thiếu khóa/đặc trưng hoặc không mang cờ buyer. Không in giá trị user IDs; không phân tích phân phối, cụm hoặc hành vi marketing.

Nếu bảng thay đổi, các kiểm tra này có thể không khớp. Điều đó cần được giải thích, không tự sửa mốc P0-T1 cho khớp với file mới.


In [ ]:
result = con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(*) FILTER (WHERE user_id IS NULL) AS missing_user_rows,
        COUNT(*) FILTER (WHERE is_buyer IS DISTINCT FROM TRUE) AS non_buyer_rows,
        COUNT(*) FILTER (WHERE
            recency_days IS NULL OR frequency IS NULL OR monetary IS NULL
            OR sessions_30d IS NULL OR views_30d IS NULL
        ) AS missing_feature_rows
    FROM buyers
""")
names = [column[0] for column in result.description]
summary = dict(zip(names, result.fetchone()))
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 5. So sánh với lần kiểm kê P0-T1
Mốc dưới được chép từ mục `data/features/buyer_features.parquet` trong `results/p0_t1/audit.json` (kiểm kê ngày 12/11/2026). Bảng gốc có **92.652 dòng, 23 cột, 2.266.870 bytes**.

Mốc được đặt ngay trong notebook để bản tải lên Colab không cần tải thêm manifest hoặc truy cập repo riêng. Chỉ cập nhật mốc khi có một lần kiểm kê được ghi nhận, không sửa để che một kết quả lệch.


In [ ]:
EXPECTED = {
    "source": "results/p0_t1/audit.json: data/features/buyer_features.parquet",
    "rows": 92652,
    "bytes": 2266870,
    "sha256": "7ffef97eb85a3bae7280d42694ba0dfc4992cf81de53826e47a73252e8ebcd3a",
    "schema": {
        "user_id": "BIGINT",
        "is_buyer": "BOOLEAN",
        "customer_cohort": "VARCHAR",
        "is_repeat_buyer": "BOOLEAN",
        "recency_days": "DOUBLE",
        "frequency": "BIGINT",
        "monetary": "DOUBLE",
        "monetary_per_session": "DOUBLE",
        "items_per_purchase_session": "DOUBLE",
        "purchase_span_days": "DOUBLE",
        "days_since_last_event": "DOUBLE",
        "sessions_30d": "BIGINT",
        "views_30d": "BIGINT",
        "carts_30d": "BIGINT",
        "unique_products_30d": "BIGINT",
        "cart_sessions_30d": "BIGINT",
        "cart_to_view_ratio_30d": "DOUBLE",
        "sessions_previous_30d": "BIGINT",
        "sessions_recent_30d": "BIGINT",
        "activity_difference": "BIGINT",
        "log_activity_ratio": "DOUBLE",
        "lifetime_events": "BIGINT",
        "lifetime_sessions": "BIGINT"
    }
}
print("Mốc P0-T1 đã nạp; không tự thay đổi theo tệp vừa đọc.")


### Đối chiếu bản sao bằng SHA256
SHA256 là dấu vân tay nội dung tệp. Khi copy từ local sang Drive, hash giống mốc gốc giúp xác nhận đang đọc đúng bản sao. Hash khớp không chứng minh cách làm sạch đã đúng hoặc dữ liệu được nghiệm thu cho nghiên cứu.

Ô này chỉ đọc lần lượt từng phần tệp và in hash, không xuất bảng khách hàng. Nếu hash khác, dừng và kiểm tra file đã chuyển; không tắt điều kiện kiểm tra.


In [ ]:
digest = hashlib.sha256()
with DATA_FILE.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
actual_sha256 = digest.hexdigest()
print("SHA256:", actual_sha256)


In [ ]:
checks = {
    "row_count_matches_P0_T1": summary["rows"] == EXPECTED["rows"],
    "schema_matches_P0_T1": actual_schema == EXPECTED["schema"],
    "file_size_matches_P0_T1": DATA_FILE.stat().st_size == EXPECTED["bytes"],
    "sha256_matches_P0_T1": actual_sha256 == EXPECTED["sha256"],
    "one_row_per_user": summary["unique_users"] == summary["rows"],
    "no_missing_user": summary["missing_user_rows"] == 0,
    "buyer_flag_only": summary["non_buyer_rows"] == 0,
    "five_features_not_missing": summary["missing_feature_rows"] == 0,
}
print(json.dumps(checks, ensure_ascii=False, indent=2))
con.close()

if not all(checks.values()):
    raise ValueError("Tệp chưa khớp mốc P0-T1. Dừng để kiểm tra, không đánh dấu đạt.")

print("Đã đọc và đối chiếu đúng tệp ứng viên P0-T1 trong phiên chạy này.")
print("P0-T3 vẫn chờ cả hai sinh viên chạy lại và giải thích kết quả trên Colab.")


### Đọc kết quả và hiểu giới hạn
Kết quả mong đợi của đúng bản gốc: 92.652 dòng, 23 cột, 92.652 users riêng biệt; các số đếm thiếu dữ liệu ở trên bằng 0; tám điều kiện đối chiếu đều đúng. Đó chỉ là kiểm tra đọc tệp và bản sao, không phải kết quả phân cụm.

Các quy tắc buyer, price, session, rounding và loại trùng 7/9 trường vẫn chưa chốt. Model/scaler/labels/metrics 7D không dùng cho MVP 5D. February không phải untouched holdout. Bảng đặc trưng không có event_time để tự kiểm tra lại cutoff từ từng event, nên bài này không thay thế kiểm tra dữ liệu nguồn.

**Trước khi nghiệm thu:** mỗi thành viên khởi động lại phiên chạy, chạy từ ô đầu tới ô cuối, tự xác nhận mount Drive và giải thích được schema/row count/hash. Không ghi PASS Colab chỉ dựa trên một lần chạy local.


## 6. Bài tập tự kiểm tra
Trả lời trước khi mở phần gợi ý:

1. Google Colab khác Jupyter chạy local như thế nào?
2. Tại sao dùng DuckDB thay vì đọc tất cả events vào pandas?
3. Schema, row count và feature table có ý nghĩa gì?

<details><summary>Gợi ý để tự kiểm tra sau khi trả lời</summary>

1. Colab chạy notebook trong runtime do Google cung cấp và cần kết nối lại Drive khi cần; Jupyter local dùng môi trường/tài nguyên máy của bạn. Cùng code có thể gặp khác biệt thư viện, đường dẫn hoặc quyền truy cập.
2. DuckDB cho phép truy vấn và chọn cột trực tiếp từ Parquet. Chưa cần đưa hàng triệu event vào pandas để chỉ kiểm tra tên cột và số dòng. pandas phù hợp khi đã thu hẹp thành bảng khách hàng.
3. Schema là cấu trúc cột/kiểu; row count là số dòng; feature table chứa các đặc trưng, ở đây một dòng/user. Nhiều cột trong bảng không có nghĩa đưa tất cả chúng vào K-Means.

</details>

**Nguồn hướng dẫn chính thức:** [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Colab: làm việc với tệp và Drive](https://colab.research.google.com/notebooks/io.ipynb), [DuckDB: đọc Parquet](https://duckdb.org/docs/stable/data/parquet/overview).

**Kết thúc P0-T3.** Chưa triển khai EDA, P0-T4 hoặc P1.


## 7. P2-T2 — Chất lượng dữ liệu toàn bộ observation

**Research Question:** dữ liệu có vấn đề nào có thể ảnh hưởng RFM + Engagement? Phần 1–6 được giữ nguyên từ bài thiết lập P0-T3. Phần này dùng kết quả tổng hợp của bốn CSV, không lấy bảng buyer cũ làm dữ liệu kiểm tra chất lượng.

**Theory / Why:** một dòng là event. Missing, loại event, duplicate, price, ID/session và timestamp phải có mẫu số rõ; số purchase rows không phải số đơn. Script `scripts/assess_quality.py` thực hiện toàn bộ truy vấn, notebook giúp đọc bảng/hình và học cách giải thích.

Để chạy phần này, mở notebook cùng bản repo có results/p2_t2 (tại root repo hoặc thư mục notebooks). Đọc bảng tổng hợp không cần raw hoặc pandas. Muốn tự quét lại thì cần bốn raw CSV và DuckDB/Matplotlib; chỉ bật tùy chọn chạy toàn bộ khi chủ động, không ghi đè output đã có. Chưa kiểm chứng lần chạy P2-T2 trên Colab.


In [1]:
from pathlib import Path
import csv

QUALITY_REPO = Path.cwd()
if not (QUALITY_REPO / "results/p2_t2/quality_summary.csv").is_file():
    QUALITY_REPO = QUALITY_REPO.parent
QUALITY_RESULTS = QUALITY_REPO / "results/p2_t2"
if not QUALITY_RESULTS.is_dir():
    raise FileNotFoundError("Cần bản repo có results/p2_t2; mở tại root repo hoặc notebooks.")
with (QUALITY_RESULTS / "quality_summary.csv").open(encoding="utf-8", newline="") as stream:
    quality_summary = list(csv.DictReader(stream))
print(f"{'Vấn đề':48} {'Số event/dòng':>14} {'Tỷ lệ (%)':>12}")
for item in quality_summary:
    print(f"{item['issue']:48} {int(item['count']):14,d} {float(item['pct']):12.4f}")


Vấn đề                                            Số event/dòng    Tỷ lệ (%)
Thiếu brand                                           6,931,209      41.9155
Thiếu category_code                                  16,259,749      98.3285
Thiếu user_id                                                 0       0.0000
Thiếu user_session                                        3,543       0.0214
user_id sai dạng số nguyên hoặc vượt BIGINT                   0       0.0000
Session không đúng dạng UUID                                  0       0.0000
Event dùng token session chung nhiều user                 1,389       0.0084
Giá âm                                                       94       0.0006
Giá bằng 0                                               32,903       0.1990
Thiếu giá                                                     0       0.0000
Giá không parse được                                          0       0.0000
Giá không hữu hạn                                             0       0.0000

### Output / Interpretation — các phát hiện chính

Đã quét **16.536.158 event**; missing category_code 98,33%, brand 41,92%, session 3.543 event. Không có purchase thiếu session trong raw này. Purchase có 91 giá âm và 1 giá 0 trên tổng 1.045.014 purchase events. Không có timestamp parse lỗi/ngoài observation/sai tháng và toàn bộ có UTC.

Tỷ lệ ở từng dòng có thể khác mẫu số: dòng purchase dùng số purchase events, giá cao dùng purchase dương, các dòng còn lại dùng tổng event. Không cộng các tỷ lệ thành tổng vì một event có thể có nhiều vấn đề.

![Missing theo cột](../results/p2_t2/figures/missing.png)

Category_code thiếu nhiều; cột session rất nhỏ trên cùng thang nhưng vẫn cần quy tắc đếm phiên. Chưa tự bỏ cột hoặc event.

![Loại sự kiện](../results/p2_t2/figures/event_types.png)

View nhiều nhất, nhưng đây là số event trước loại trùng; không gọi là tỷ lệ khách mua hoặc funnel chuyển đổi.

![Price](../results/p2_t2/figures/price.png)

Hai ô dùng cùng 60 khoảng và trục số lượng log. Nhóm price>0 chỉ để so phạm vi giá, không phải dữ liệu đã làm sạch. Giá cao có thể hợp lệ; không tự loại.

![Duplicate 7 và 9 trường](../results/p2_t2/figures/duplicates.png)

Cả hai khóa cho 868.808 dòng dư (5,2540%); 1.695.067 dòng nằm trong 826.259 nhóm lặp. Đây là phép so văn bản gốc, chưa phải quyết định loại trùng hoặc tái tạo ETL cũ sau ép kiểu. Màu đậm nhấn khóa đang xem, không phải phương án thắng.


In [2]:
with (QUALITY_RESULTS / "quality_details.csv").open(encoding="utf-8", newline="") as stream:
    quality_details = list(csv.DictReader(stream))
print("Missing theo tháng (mẫu số = số event từng tháng):")
for item in quality_details:
    if item["scope"] == "month" and item["metric"] in {
        "missing_brand", "missing_category_code", "missing_user_session"
    }:
        print(item["month"], item["metric"], int(item["value"]),
              "/", int(item["denominator"]), f"{float(item['pct']):.4f}%")


Missing theo tháng (mẫu số = số event từng tháng):
2019-10 missing_category_code 4034806 / 4102283 98.3551%
2019-10 missing_brand 1659261 / 4102283 40.4473%
2019-10 missing_user_session 637 / 4102283 0.0155%
2019-11 missing_category_code 4560089 / 4635837 98.3660%
2019-11 missing_brand 1986029 / 4635837 42.8408%
2019-11 missing_user_session 813 / 4635837 0.0175%
2019-12 missing_category_code 3474821 / 3533286 98.3453%
2019-12 missing_brand 1510289 / 3533286 42.7446%
2019-12 missing_user_session 779 / 3533286 0.0220%
2020-01 missing_category_code 4190033 / 4264752 98.2480%
2020-01 missing_brand 1775630 / 4264752 41.6350%
2020-01 missing_user_session 1314 / 4264752 0.0308%


In [3]:
# Chỉ bật khi muốn tự quét lại raw; output mới để giữ kết quả đã có.
RUN_FULL_ASSESSMENT = False
if RUN_FULL_ASSESSMENT:
    import subprocess
    import sys
    subprocess.run([
        sys.executable, "-B", str(QUALITY_REPO / "scripts/assess_quality.py"),
        "--data-root", str(DATA_ROOT),
        "--output", "results/p2_t2_student_recheck",
    ], cwd=QUALITY_REPO, check=True)
else:
    print("Chỉ đọc kết quả tổng hợp; chưa chạy lại toàn bộ raw.")


Chỉ đọc kết quả tổng hợp; chưa chạy lại toàn bộ raw.


### Verification / Limitations / Bài tập

Script đã đối chiếu tổng dòng theo tháng/loại, tổng tần số khóa, phân nhóm giá/thời gian và histogram; năm test phần mềm kiểm tra công thức và toàn luồng xuất bảng/hình trên ví dụ riêng. Hash tháng 12 trước/sau giống nhau và metadata bốn raw không đổi. Hash local chưa chứng minh nguồn tải; không dùng kết quả test thay dữ liệu nghiên cứu.

Chỉ các ô mới phần 7 được chạy bằng Python local để kiểm tra đọc bảng; chưa khởi động lại/chạy toàn notebook trên Colab ở task này. Hai sinh viên chạy trên bản sao riêng theo quy trình nhóm; không sửa đè notebook của nhau.

1. Vì sao tỷ lệ lỗi giá ở purchase dùng mẫu số khác tỷ lệ thiếu brand?
2. Một khóa lặp ba lần có bao nhiêu dòng dư? Hai cột duplicate bằng nhau có cho phép tự chọn policy không?
3. Vì sao thiếu session chưa bắt buộc loại Views, và giá cao chưa phải lỗi?

Gợi ý và công thức ở LEARNING_GUIDE phần 18. P2-T3 mới chọn quy tắc; dừng chờ nghiệm thu P2-T2.
